In [1]:
# pip install datasets sentencepiece sacrebleu tokenizers

In [2]:
%cd /content/drive/MyDrive/Colab\ Notebooks/transformer

/content/drive/MyDrive/Colab Notebooks/transformer


In [3]:
import os
import json
import numpy as np
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer

In [4]:
dataset = load_dataset("Helsinki-NLP/opus_books", "de-en")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

de-en/train-00000-of-00001.parquet:   0%|          | 0.00/8.80M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51467 [00:00<?, ? examples/s]

In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 51467
    })
})


In [6]:
for i in range(1, 11):
    print(dataset["train"]["translation"][i])

{'de': 'Jane Eyre', 'en': 'Jane Eyre'}
{'de': 'Charlotte Bronte', 'en': 'Charlotte Bronte'}
{'de': 'Erstes Kapitel', 'en': 'CHAPTER I'}
{'de': 'Es war ganz unmöglich, an diesem Tage einen Spaziergang zu machen.', 'en': 'There was no possibility of taking a walk that day.'}
{'de': 'Am Morgen waren wir allerdings während einer ganzen Stunde in den blätterlosen, jungen Anpflanzungen umhergewandert; aber seit dem Mittagessen – Mrs. Reed speiste stets zu früher Stunde, wenn keine Gäste zugegen waren – hatte der kalte Winterwind so düstere, schwere Wolken und einen so durchdringenden Regen heraufgeweht, daß von weiterer Bewegung in frischer Luft nicht mehr die Rede sein konnte.', 'en': 'We had been wandering, indeed, in the leafless shrubbery an hour in the morning; but since dinner (Mrs. Reed, when there was no company, dined early) the cold winter wind had brought with it clouds so sombre, and a rain so penetrating, that further out-door exercise was now out of the question.'}
{'de': "Ich 

In [7]:
# First take 50% of the full dataset
half = dataset['train'].train_test_split(test_size=0.5, seed=42)
half_data = half['test']  # 50% of total

# Now do the normal train/val/test splits on the half
split1 = half_data.train_test_split(test_size=0.1, seed=42)
split2 = split1['train'].train_test_split(test_size=0.056, seed=42)

train_data = split2['train']
val_data   = split2['test']
test_data  = split1['test']

print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")

Train: 21863
Val:   1297
Test:  2574


# Save Dataset

In [8]:
os.makedirs('data', exist_ok=True)

train_data.to_json(f'data/train.jsonl')
val_data.to_json(f'data/val.jsonl')
test_data.to_json(f'data/test.jsonl')

Creating json from Arrow format:   0%|          | 0/22 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

797035

# Load Dataset

In [9]:
BASE = '/content/drive/MyDrive/Colab Notebooks/transformer'

train_data = load_dataset('json', data_files=f'{BASE}/data/train.jsonl')['train']
val_data   = load_dataset('json', data_files=f'{BASE}/data/val.jsonl')['train']
test_data  = load_dataset('json', data_files=f'{BASE}/data/test.jsonl')['train']

print(f"Train: {len(train_data):,}")
print(f"Val:   {len(val_data):,}")
print(f"Test:  {len(test_data):,}")

en_sentences = [s['translation']['en'] for s in train_data]
de_sentences = [s['translation']['de'] for s in train_data]
all_sentences = en_sentences + de_sentences

print(f"\nTotal sentences (en+de combined): {len(all_sentences):,}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 21,863
Val:   1,297
Test:  2,574

Total sentences (en+de combined): 43,726


# BPE TOKENIZATION

In [10]:
# BPE TOKENIZATION ANALYSIS, Word-level vocab is misleading. BPE splits rare words into subwords.

# Train a BPE tokenizer on actual data
print("Training BPE tokenizer on the data...")

# Save sentences to a temp file (tokenizers library needs a file)
tmp_file = '/tmp/all_sentences.txt'
with open(tmp_file, 'w', encoding='utf-8') as f:
    for s in all_sentences:
        f.write(s.strip() + '\n')

# Train BPE tokenizer
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=[tmp_file],
    vocab_size=37000,
    min_frequency=2,
    special_tokens=["<pad>", "<sos>", "<eos>", "<unk>"]
)

print("BPE tokenizer trained.")
print(f"Actual vocab size: {tokenizer.get_vocab_size():,}")

Training BPE tokenizer on the data...
BPE tokenizer trained.
Actual vocab size: 37,000


The Ġ symbol is just the byte-level representation of a "space" before a word

In [11]:
encoded = tokenizer.encode("How are you. What's going on?")
print(encoded.tokens)

['How', 'Ġare', 'Ġyou', '.', 'ĠWhat', "'s", 'Ġgoing', 'Ġon', '?']


# SEQUENCE LENGTH AFTER BPE TOKENIZATION

In [12]:
# BPE tokens != words. "unfortunately" might become ["un", "fort", "unately"]
# Lengths will be longer than word-level counts.

print("Measuring BPE token lengths...")

bpe_lengths = [len(tokenizer.encode(s).ids) for s in all_sentences]

print("=" * 50)
print("SEQUENCE LENGTH STATS (after BPE tokenization)")
print("=" * 50)
print(f"  min:    {min(bpe_lengths)}")
print(f"  max:    {max(bpe_lengths)}")
print(f"  mean:   {np.mean(bpe_lengths):.1f}")
print(f"  median: {np.median(bpe_lengths):.1f}")
print(f"  90th %: {np.percentile(bpe_lengths, 90):.1f}")
print(f"  95th %: {np.percentile(bpe_lengths, 95):.1f}")
print(f"  99th %: {np.percentile(bpe_lengths, 99):.1f}")
print(f"  max:    {max(bpe_lengths)}")

print(f"\nSentences fitting within max_seq_len (BPE tokens):")
for max_len in [50, 80, 100, 128, 150, 200, 256, 500]:
    count = sum([l <= max_len for l in bpe_lengths])
    print(f"  Sentences ≤ {max_len} tokens: {count} / {len(bpe_lengths)}")

Measuring BPE token lengths...
SEQUENCE LENGTH STATS (after BPE tokenization)
  min:    1
  max:    569
  mean:   26.9
  median: 21.0
  90th %: 54.0
  95th %: 69.0
  99th %: 108.0
  max:    569

Sentences fitting within max_seq_len (BPE tokens):
  Sentences ≤ 50 tokens: 38521 / 43726
  Sentences ≤ 80 tokens: 42386 / 43726
  Sentences ≤ 100 tokens: 43127 / 43726
  Sentences ≤ 128 tokens: 43500 / 43726
  Sentences ≤ 150 tokens: 43604 / 43726
  Sentences ≤ 200 tokens: 43680 / 43726
  Sentences ≤ 256 tokens: 43702 / 43726
  Sentences ≤ 500 tokens: 43722 / 43726


In [13]:
# we will use max_seq_len = 128, vocab_size = 37000

In [14]:
tokenizer.save("bpe_tokenizer.json")